In [1]:
print("hola")

hola


In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. CONFIGURACIÓN: CAMBIA SOLO ESTE NOMBRE DE ARCHIVO
#    Puede tener extensión .html o .xls (es HTML igual)
# ============================================================
archivo_html = Path("201 - Seguimiento Insumos.xls")
# archivo_html = Path("SeguimientoInsumos.xls")  # también sirve

print(f"[DEBUG] Leyendo tabla HTML desde: {archivo_html}")

# ============================================================
# 2. LEER LA TABLA USANDO LAS FILAS 3 Y 4 DEL THEAD COMO HEADER
#    (0-based dentro de la tabla: filas con títulos grandes y Cantidad/Valor)
# ============================================================
dataframe_bruto = pd.read_html(archivo_html, header=[3, 4])[0]

print("[DEBUG] Columnas MultiIndex leídas:")
print(dataframe_bruto.columns)

# ============================================================
# 3. APLANAR EL MULTIINDEX A NOMBRES TIPO:
#    - Descripcion, Tipo, Codigo, UM
#    - Cantidad_Presupuesto, Valor_Presupuesto, Cantidad_Proyectado, ...
# ============================================================
nuevos_nombres_columnas = []

for nombre_nivel_0, nombre_nivel_1 in dataframe_bruto.columns:
    texto_nivel_0 = str(nombre_nivel_0).strip() if pd.notna(nombre_nivel_0) else ""
    texto_nivel_1 = str(nombre_nivel_1).strip() if pd.notna(nombre_nivel_1) else ""

    # Columnas "simples" sin subheader
    if texto_nivel_1 == "" or texto_nivel_0 in ["Descripción", "Tipo", "Código", "UM"]:
        nombre = texto_nivel_0 or texto_nivel_1
    else:
        # Columnas de bloques: Cantidad_Presupuesto, Valor_Proyectado, etc.
        nombre = f"{texto_nivel_1}_{texto_nivel_0}"

    # Normalizar un poco el nombre (sin tildes ni espacios)
    nombre = (
        nombre.replace(" ", "_")
              .replace("ó", "o").replace("Ó", "O")
              .replace("í", "i").replace("Í", "I")
              .replace("á", "a").replace("Á", "A")
              .replace("é", "e").replace("É", "E")
              .replace("ú", "u").replace("Ú", "U")
              .replace("<", "MenorQue")
              .replace(">", "MayorQue")
    )

    nuevos_nombres_columnas.append(nombre)

dataframe_bruto.columns = nuevos_nombres_columnas

print("[DEBUG] Columnas aplanadas:")
print(dataframe_bruto.columns.tolist())

dataframe = dataframe_bruto.copy()

# ============================================================
# 4. LIMPIEZA BÁSICA: ASEGURAR COLUMNA 'Descripcion'
# ============================================================
if "Descripcion" not in dataframe.columns:
    raise ValueError(
        "No se encontró la columna 'Descripcion'. "
        "Revisa el header=[3,4] o imprime dataframe_bruto.head()."
    )

serie_descripcion = dataframe["Descripcion"].astype("string")

# ============================================================
# 5. CLASIFICAR FILAS EN:
#    - Capitulo: '1-PRELIMINARES', '2-CIMENTACION', ...
#    - Subnivel: '1.002-Localización...', '1.003-Cerramiento...', ...
#    - Insumo:   todo lo demás (descripciones "normales")
# ============================================================

# Patrón de capítulo: número-guion-texto
patron_capitulo = r"^\d+-"

# Patrón de subnivel: número.punto número-guion-texto
patron_subnivel = r"^\d+\.\d+-"

# Marcar capítulos
dataframe["Capitulo"] = np.where(
    serie_descripcion.str.match(patron_capitulo, na=False),
    serie_descripcion,
    np.nan,
)

# Marcar subniveles
dataframe["Subnivel"] = np.where(
    serie_descripcion.str.match(patron_subnivel, na=False),
    serie_descripcion,
    np.nan,
)

# Marcar insumos (ni capítulo ni subnivel, pero con descripción)
es_capitulo = serie_descripcion.str.match(patron_capitulo, na=False)
es_subnivel = serie_descripcion.str.match(patron_subnivel, na=False)

dataframe["Insumo"] = np.where(
    (~es_capitulo & ~es_subnivel & serie_descripcion.notna()),
    serie_descripcion,
    np.nan,
)

# Heredar hacia abajo el último capítulo y subnivel encontrados
dataframe["Capitulo"] = dataframe["Capitulo"].ffill()
dataframe["Subnivel"] = dataframe["Subnivel"].ffill()

print("[DEBUG] Primeras filas con Descripcion / Capitulo / Subnivel / Insumo:")
print(
    dataframe[["Descripcion", "Capitulo", "Subnivel", "Insumo"]]
    .head(20)
    .to_string(index=False)
)

# ============================================================
# 6. SEPARAR CÓDIGO Y NOMBRE DE CAPÍTULO Y SUBNIVEL
#    - Capitulo: '1-PRELIMINARES' -> codigo='1', nombre='PRELIMINARES'
#    - Subnivel: '1.002-Cerramiento...' -> codigo='1.002', nombre='Cerramiento...'
# ============================================================

# Capítulo
partes_capitulo = dataframe["Capitulo"].astype("string").str.extract(r"^(\d+)-\s*(.*)$")
dataframe["Capitulo_codigo"] = partes_capitulo[0]
dataframe["Capitulo_nombre"] = partes_capitulo[1]

# Subnivel
partes_subnivel = dataframe["Subnivel"].astype("string").str.extract(r"^(\d+\.\d+)-\s*(.*)$")
dataframe["Subnivel_codigo"] = partes_subnivel[0]
dataframe["Subnivel_nombre"] = partes_subnivel[1]

# ============================================================
# 7. OPCIONAL: FILTRAR SOLO INSUMOS "REALES"
# ============================================================
dataframe_insumos = dataframe[dataframe["Insumo"].notna()].copy()

print("[DEBUG] Ejemplo de filas solo_insumos:")
print(
    dataframe_insumos[
        ["Capitulo_codigo", "Capitulo_nombre",
         "Subnivel_codigo", "Subnivel_nombre",
         "Insumo"]
    ]
    .head(20)
    .to_string(index=False)
)

# ============================================================
# 8. GUARDAR RESULTADOS A EXCEL
# ============================================================
salida_completa = archivo_html.with_name(archivo_html.stem + "_limpio_completo.xlsx")
salida_insumos = archivo_html.with_name(archivo_html.stem + "_solo_insumos.xlsx")

dataframe.to_excel(salida_completa, index=False)
dataframe_insumos.to_excel(salida_insumos, index=False)

print(f"[DEBUG] Archivo completo guardado en: {salida_completa}")
print(f"[DEBUG] Archivo solo insumos guardado en: {salida_insumos}")


[DEBUG] Leyendo tabla HTML desde: 201 - Seguimiento Insumos.xls
[DEBUG] Columnas MultiIndex leídas:
MultiIndex([('Descripción',     'COSTOS DIRECTOS'),
            (       'Tipo',  'Unnamed: 1_level_1'),
            (     'Código',  'Unnamed: 2_level_1'),
            (         'UM',  'Unnamed: 3_level_1'),
            (   'Cantidad',  'Unnamed: 4_level_1'),
            (      'Valor',      '11,104,591,989'),
            (   'Cantidad',  'Unnamed: 6_level_1'),
            (      'Valor',      '11,815,713,813'),
            (   'Cantidad',  'Unnamed: 8_level_1'),
            (      'Valor',        '-711,121,824'),
            (   'Cantidad', 'Unnamed: 10_level_1'),
            (      'Valor',       '9,124,046,860'),
            (   'Cantidad', 'Unnamed: 12_level_1'),
            (      'Valor',       '2,843,400,279'),
            (   'Cantidad', 'Unnamed: 14_level_1'),
            (      'Valor',      '11,967,447,139'),
            (   'Cantidad', 'Unnamed: 16_level_1'),
            (   

KeyboardInterrupt: 

In [9]:
dataframe

,Descripcion,Tipo,Codigo,UM,Unnamed:_4_level_1_Cantidad,"11,104,591,989_Valor",Unnamed:_6_level_1_Cantidad,"11,815,713,813_Valor",Unnamed:_8_level_1_Cantidad,"-711,121,824_Valor",...,Unnamed:_22_level_1_Valor,Unnamed:_23_level_1_ConsuMenorQueProy,Unnamed:_24_level_1_AsegMenorQueProy,Capitulo,Subnivel,Insumo,Capitulo_codigo,Capitulo_nombre,Subnivel_codigo,Subnivel_nombre
0,1-PRELIMINARES,NaN,NaN,NaN,NaN,171163790,NaN,136522928,NaN,34640863,...,NaN,✓,✓,1-PRELIMINARES,NaN,NaN,1,PRELIMINARES,<NA>,<NA>
1,1.001-Localización y replanteo,NaN,NaN,NaN,699.30,3718446,699.30,3718450,0.00,-4,...,NaN,✓,✓,1-PRELIMINARES,1.001-Localización y replanteo,NaN,1,PRELIMINARES,1.001,Localización y replanteo
2,comisión de topografía,O,1933,di,10.00,3718446,10.00,3718450,0.00,-4,...,100.00%,NaN,NaN,1-PRELIMINARES,1.001-Localización y replanteo,comisión de topografía,1,PRELIMINARES,1.001,Localización y replanteo
3,1.002-Cerramiento en lona altura=2.00m,NaN,NaN,NaN,29.88,697419,0.00,0,29.88,697419,...,NaN,✓,✓,1-PRELIMINARES,1.002-Cerramiento en lona altura=2.00m,NaN,1,PRELIMINARES,1.002,Cerramiento en lona altura=2.00m
4,mano de obra preliminares ug,O,8625,un,29.88,239040,0.00,0,29.88,239040,...,0.00%,NaN,NaN,1-PRELIMINARES,1.002-Cerramiento en lona altura=2.00m,mano de obra preliminares ug,1,PRELIMINARES,1.002,Cerramiento en lona altura=2.00m
5,concreto 2500 psi grava común,M,1952,m3,0.30,112503,0.00,0,0.30,112503,...,0.00%,NaN,NaN,1-PRELIMINARES,1.002-Cerramiento en lona altura=2.00m,concreto 2500 psi grava común,1,PRELIMINARES,1.002,Cerramiento en lona altura=2.00m
6,listón 0.02x0.01x3m,M,3626,un,14.94,28984,0.00,0,14.94,28984,...,0.00%,NaN,NaN,1-PRELIMINARES,1.002-Cerramiento en lona altura=2.00m,listón 0.02x0.01x3m,1,PRELIMINARES,1.002,Cerramiento en lona altura=2.00m
7,"puntilla con cabeza de 1-1/2"" a 4"" (p)",M,5022,lb,16.43,28760,0.00,0,16.43,28760,...,0.00%,NaN,NaN,1-PRELIMINARES,1.002-Cerramiento en lona altura=2.00m,"puntilla con cabeza de 1-1/2"" a 4"" (p)",1,PRELIMINARES,1.002,Cerramiento en lona altura=2.00m
8,tela protección verde x 2.10m,M,6585,ml,29.88,109301,0.00,0,29.88,109301,...,0.00%,NaN,NaN,1-PRELIMINARES,1.002-Cerramiento en lona altura=2.00m,tela protección verde x 2.10m,1,PRELIMINARES,1.002,Cerramiento en lona altura=2.00m
9,vara corredor 3m,M,7349,un,14.94,178832,0.00,0,14.94,178832,...,0.00%,NaN,NaN,1-PRELIMINARES,1.002-Cerramiento en lona altura=2.00m,vara corredor 3m,1,PRELIMINARES,1.002,Cerramiento en lona altura=2.00m


In [ ]:
file_path = Path("201 - Seguimiento Insumos.xls")


# Esto funciona si el .xls es realmente HTML por dentro
print("[DEBUG] Intentando leer como HTML...")
tabla_lista = pd.read_html(file_path)  # puede haber varias tablas, nos quedamos con la primera
df_raw = tabla_lista[0]

print("[DEBUG] Primeras columnas tal cual vienen:")
df_raw.head()


[DEBUG] Intentando leer como HTML...
[DEBUG] Primeras columnas tal cual vienen:


MultiIndex([( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025',

In [ ]:
file_path = Path("201 - Seguimiento Insumos.xls")


# Esto funciona si el .xls es realmente HTML por dentro
print("[DEBUG] Intentando leer como HTML...")
tabla_lista = pd.read_html(file_path)  # puede haber varias tablas, nos quedamos con la primera
df_raw = tabla_lista[0]

print("[DEBUG] Primeras columnas tal cual vienen:")
df_raw.head()

df_raw.columns


MultiIndex([( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ( 'Proyecto: Edificio Sangregado', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025', ...),
            ('Fecha de Impresión: 20/11/2025',

In [24]:
from pathlib import Path
import pandas as pd
import numpy as np

file_path = Path("201 - Seguimiento Insumos.xls")

print("[DEBUG] Leyendo tabla sin encabezado...")
tabla_lista = pd.read_html(file_path, header=None)
df_html = tabla_lista[0]

# 1) Localizar la fila donde está la palabra "Descripción"
fila_header = df_html[df_html.iloc[:, 0] == "Descripción"].index[0]
print(f"[DEBUG] Fila de header encontrada en índice: {fila_header}")

# 2) Volver a leer usando esa fila y la siguiente como header (multi-nivel)
df_raw = pd.read_html(file_path, header=[fila_header, fila_header + 1])[0]

print("[DEBUG] Columnas MultiIndex correctas:")
print(df_raw.columns)

# 3) Ahora sí puedes acceder a Proyectado → Cantidad y Valor
serie_proyectado_cantidad = df_raw[("Proyectado", "Cantidad")]
serie_proyectado_valor    = df_raw[("Proyectado", "Valor")]

print("[DEBUG] Proyectado_Cantidad (primeras filas):")
print(serie_proyectado_cantidad.head())

print("[DEBUG] Proyectado_Valor (primeras filas):")
print(serie_proyectado_valor.head())


[DEBUG] Leyendo tabla sin encabezado...


IndexError: index 0 is out of bounds for axis 0 with size 0

In [21]:
for i in range(df_raw.shape[1]):
    display(df_raw.iloc[1, i])

'1-PRELIMINARES'

nan

nan

nan

nan

...

np.float64(nan)

np.float64(-0.0)

nan

'✓'

'✓'